# 第 2 章 提示词与消息（Prompts & Messages）

**学习目标**
- 理解 LangChain 中 4 种核心消息类型及其在对话中的作用
- 熟练使用 `ChatPromptTemplate` 编写带变量的提示词模板
- 掌握 `MessagesPlaceholder` 插入对话历史 / 少样本示例
- 会用 few-shot（少样本）提示显著改变模型行为
- 理解 `prompt | llm | parser` 这种"链"的雏形（第 4 章展开）

> 前置条件：已完成 chapter01，确保 `.env` 中配置了 `API_KEY_ZHIPU` 和 `API_URL_ZHIPU`。

In [1]:
# 环境准备：加载 .env 并初始化模型（本系列所有章节通用）
import os
from dotenv import load_dotenv

load_dotenv()  # 读取项目根目录下的 .env 文件

from langchain_openai import ChatOpenAI

# 智谱 GLM 提供 OpenAI 兼容接口，直接用 ChatOpenAI 指向智谱的 base_url 即可
llm = ChatOpenAI(
    model="glm-4.5-air",  # 可换成你账号可用的其他模型（改模型名即可，无需改其他代码）
    temperature=0.6,
    api_key=os.getenv("API_KEY_ZHIPU"),
    base_url=os.getenv("API_URL_ZHIPU"),  # 形如 https://open.bigmodel.cn/api/paas/v4/
)

print("模型就绪:", llm.model_name)
# 注意：本章会发起若干次真实 API 调用，会产生少量费用；练习请使用短文本。

模型就绪: glm-4.5-air


## 1. 消息类型（Messages）

LangChain 中一次对话就是 `BaseMessage` 列表，最常用的 4 种：

| 类型 | 角色 | 用途 |
|---|---|---|
| `SystemMessage` | system | 设定身份、规则、风格，通常放最前面且只有一条 |
| `HumanMessage` | user | 用户输入 |
| `AIMessage` | assistant | 模型的历史回复（多轮对话时回传给模型） |
| `ToolMessage` | tool | 工具执行结果（第 5 章详解） |

> **关键点**：模型本身是**无状态**的。所谓"记住上下文"，就是每次请求时把历史消息一起发过去。

In [2]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 多轮对话：把历史消息原样带上，模型就知道上下文
messages = [
    SystemMessage(content="你是一位面试官，语气专业但友善"),
    HumanMessage(content="你好，我想应聘 Python 后端岗位"),
    AIMessage(content="你好！很高兴认识你。请先介绍一下你做过的最有挑战的项目吧。"),
    HumanMessage(content="我做过一个日均百万请求的推荐服务"),
]

response = llm.invoke(messages)
print(response.content)


感谢分享这个经验。日均百万请求的推荐服务是一个相当有挑战性的项目。能否请您详细介绍一下：

1. 您在这个项目中担任的角色和主要职责是什么？
2. 推荐服务的整体架构是怎样的？采用了哪些关键技术？
3. 如何处理高并发场景下的性能优化？
4. 在这个项目中，您遇到的最大挑战是什么，是如何解决的？
5. 如何评估推荐系统的效果和性能指标？

这些信息将帮助我更好地了解您的技术能力和项目经验。


`response` 是一个 `AIMessage` 对象，除了 `content` 还有几个常用属性：
- `response_metadata`：模型名、finish_reason、token 用量等
- `usage_metadata`：input_tokens / output_tokens（计费参考）

下面观察一下：

In [3]:
print("content 类型:", type(response.content))
print("回复长度:", len(response.content))
print("usage:", response.usage_metadata)
print("finish_reason:", response.response_metadata.get("finish_reason"))

content 类型: <class 'str'>
回复长度: 197
usage: {'input_tokens': 57, 'output_tokens': 249, 'total_tokens': 306, 'input_token_details': {'cache_read': 6}, 'output_token_details': {}}
finish_reason: stop


## 2. ChatPromptTemplate：提示词模板

把固定的话术和可变的"变量"分离，是工程化使用 LLM 的第一步。

```python
prompt = ChatPromptTemplate.from_messages([
    ("system", "..."),   # ("角色", "内容模板")
    ("human", "..."),
])
```

模板中用 `{变量名}` 占位，调用时用字典传值。

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一位{domain}领域的资深专家，用{style}的风格回答问题。"),
    ("human", "{question}"),
])

# 两种常用用法：

# 用法 1：format_messages() —— 只做格式化，得到消息列表，再手动传给模型
msgs = prompt.format_messages(
    domain="数据库", style="深入浅出", question="索引为什么能加快查询？"
)
print(msgs)

# 用法 2：invoke() —— 返回 ChatPromptValue，可直接接在链后面
prompt_value = prompt.invoke({
    "domain": "数据库",
    "style": "深入浅出",
    "question": "索引为什么能加快查询？",
})
print(type(prompt_value))
response = llm.invoke(prompt_value.to_messages())
print(response.content[:100])

[SystemMessage(content='你是一位数据库领域的资深专家，用深入浅出的风格回答问题。', additional_kwargs={}, response_metadata={}), HumanMessage(content='索引为什么能加快查询？', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.prompt_values.ChatPromptValue'>
# 索引如何加速查询

索引就像一本书的目录，能帮你快速找到想要的内容，而不必一页一页翻整本书。在数据库中，索引通过以下方式加速查询：

## 1. 减少数据扫描量

没有索引时，数据库需要进行**全


In [ ]:
# 更常见的写法：模板直接接上模型和输出解析器，用 | 串成一条链
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一位{domain}领域的资深专家，用{style}的风格回答问题。"),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()   # StrOutputParser 把 AIMessage 简化为纯字符串

result = chain.invoke({
    "domain": "数据库",
    "style": "深入浅出",
    "question": "索引为什么能加快查询？",
})
print(result)

> **`{}` 的坑**：模板里如果真的需要输出花括号本身（比如 JSON 示例），要写成双花括号 `{{` `}}`，否则会被当成变量。

## 3. MessagesPlaceholder：插入消息列表

当变量是一个**消息列表**（历史对话、示例）而不是字符串时，用 `MessagesPlaceholder`。它是实现"多轮对话"和"few-shot"的标配。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是助手 {name}，请保持{style}的语气。"),
    MessagesPlaceholder("history"),      # 这里会被展开成任意条历史消息
    ("human", "{question}"),
])

history = [
    HumanMessage(content="我叫小明"),
    AIMessage(content="你好，小明！很高兴见到你。"),
]

response = llm.invoke(prompt.invoke({
    "name": "小智",
    "style": "幽默",
    "history": history,
    "question": "你还记得我叫什么吗？",
}))
print(response.content)

`MessagesPlaceholder` 支持设置 `optional=True`：当调用时不传 `history` 也不会报错，非常适合"首轮对话没有历史"的场景。

```python
MessagesPlaceholder("history", optional=True)
```

## 4. Few-shot 少样本提示

在提示里给模型几个"输入→输出"示例，能显著提升分类、格式转换等任务的准确率。用 `FewShotChatMessagePromptTemplate` 可以把示例统一管理。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 准备示例
examples = [
    {"input": "今天天气真差，心情都坏了", "output": "负面"},
    {"input": "服务态度特别好，下次还来", "output": "正面"},
    {"input": "东西一般般，凑合能用", "output": "中性"},
]

# 2. 单条示例长什么样
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

# 3. 组装 few-shot 模板：自动把 examples 逐条套进 example_prompt
few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# 4. 最终提示 = 系统设定 + few-shot 示例 + 用户输入
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是评论情感分类器。只输出：正面、负面 或 中性 之一，不要输出其他内容。"),
    few_shot,
    ("human", "{input}"),
])

classifier = final_prompt | llm | StrOutputParser()

for text in ["物流速度快但是包装破损了", "性价比无敌，闭眼入", "跟描述基本一致，无惊喜也无雷"]:
    print(text, "->", classifier.invoke({"input": text}))

## 5. partial：模板的部分变量先固定

`partial()` 可以提前固定一部分变量（如当前日期、格式要求），剩下的变量留到调用时再传。

In [ ]:
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")

prompt = ChatPromptTemplate.from_messages([
    ("system", "今天是 {date}。你是 {role}。回答时请结合当前日期考虑时效性。"),
    ("human", "{question}"),
]).partial(date=today)   # date 变量已固定，之后只需传 role 和 question

chain = prompt | llm | StrOutputParser()
print(chain.invoke({"role": "旅行顾问", "question": "下周去三亚穿什么合适？"}))

---

## 练习

> 建议先自己写，写完再对照文末参考答案。每个练习都是真实 API 调用。

**练习 1：可指定目标语言的翻译助手**
- 模板变量：`target_language`（目标语言）、`text`（待翻译文本）
- system 提示要求：只输出译文，不要解释
- 用一条链完成，测试中英互译和"翻译成文言文"

**练习 2：少样本提示做"同义句改写"**
- 准备 3 个改写示例（如 "我很开心" → "我心花怒放"）
- 让模型对 "这道题太难了" 生成改写
- 观察：把示例从 3 个减到 0 个，风格差异有多大？

**练习 3：会议记录整理器**
- 给一段口语化的会议记录（练习里有示例）
- 用 system 提示固定输出格式（参会人 / 结论 / 待办事项三段）
- 用 `partial` 固定"整理日期"为今天

---

## 参考答案（先自己写再看！）

In [ ]:
# 练习 1 参考答案：翻译助手
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是专业翻译。将用户输入翻译成{target_language}，只输出译文，不要任何解释。"),
    ("human", "{text}"),
])
translate_chain = translate_prompt | llm | StrOutputParser()

print(translate_chain.invoke({"target_language": "英语", "text": "工欲善其事，必先利其器"}))
print(translate_chain.invoke({"target_language": "文言文", "text": "加油干，别放弃"}))

In [ ]:
# 练习 2 参考答案：few-shot 同义句改写
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

rewrite_examples = [
    {"input": "我很开心", "output": "我心花怒放"},
    {"input": "他很生气", "output": "他怒不可遏"},
    {"input": "今天很热", "output": "今日骄阳似火"},
]
example_prompt = ChatPromptTemplate.from_messages([("human", "{input}"), ("ai", "{output}")])
few_shot = FewShotChatMessagePromptTemplate(example_prompt=example_prompt, examples=rewrite_examples)

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "将用户输入改写为更生动的表达，只输出改写结果。"),
    few_shot,
    ("human", "{input}"),
])
rewrite_chain = rewrite_prompt | llm | StrOutputParser()

print("有示例:", rewrite_chain.invoke({"input": "这道题太难了"}))

# 对照组：去掉示例
plain_chain = ChatPromptTemplate.from_messages([
    ("system", "将用户输入改写为更生动的表达，只输出改写结果。"),
    ("human", "{input}"),
]) | llm | StrOutputParser()
print("无示例:", plain_chain.invoke({"input": "这道题太难了"}))

In [ ]:
# 练习 3 参考答案：会议记录整理器
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

raw_notes = (
    "今天下午三点开的周会，老张、小李和我参加了。老张说上周销售额涨了15%，"
    "主要是直播带货带来的。小李反馈客服系统最近老崩，需要运维看看。"
    "最后说好下周三之前小李要把客服崩溃问题排查报告写出来，"
    "我负责整理这份会议纪要，老张继续盯销售数据。散会的时候都快六点了。"
)

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", '''你是会议记录整理专家。今天是 {date}。
请把口语化的会议记录整理成以下格式（Markdown）：
# 会议纪要
## 参会人
## 主要结论
## 待办事项（每条注明负责人和截止时间）'''),
    ("human", "{notes}"),
]).partial(date=datetime.now().strftime("%Y-%m-%d"))

summary_chain = summary_prompt | llm | StrOutputParser()
print(summary_chain.invoke({"notes": raw_notes}))